In [0]:
import os
import mlflow

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/1")

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

# Read test data and normalize (model was trained on normalized features)
train = spark.read.table("teams.sensorx.train_features_delta")
test = spark.read.table("teams.sensorx.test_features_delta")

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# Filter to latest 3 weeks
max_date = test_norm.agg(F.max("timeStamp")).collect()[0][0]
test_recent = test_norm.filter(F.col("timeStamp") >= F.date_sub(F.lit(max_date), 21))

# Run predictions
predictions = model.transform(test_recent)

# Join with original table to get actual fault state and deviceId
original = spark.read.table("teams.sensorx.df_sx_800_with_delta").select("timeStamp", "payload_fault_state", "properties_deviceId")

pred_with_fault = predictions.select(
    "timeStamp", "label", "prediction"
).join(original, on="timeStamp", how="left") \
 .withColumn("date", F.to_date("timeStamp"))

# One prediction per day per device
daily_results = pred_with_fault.groupBy("date", "properties_deviceId").agg(
    F.max("prediction").cast("int").alias("daily_prediction"),
    F.max(F.col("payload_fault_state").cast("int")).alias("actual_fault_state"),
    F.max("label").alias("horizon_failure"),
    F.count("*").alias("readings_count")
).orderBy("date", "properties_deviceId")

display(daily_results)

In [0]:
from pyspark.sql import Row

data = [
    Row(Trial=1,  AUC=0.6525), #Logistic Regression 1, 5.mars
    Row(Trial=2,  AUC=0.6251), #Random Forest 1, 5.mars
    Row(Trial=3,  AUC=0.5383), #Gradient Boosted Trees 1, 5.mars
    Row(Trial=4,  AUC=0.9115), #Logistic Regression 2, 8.mars
    Row(Trial=5,  AUC=0.9139), #Logistic Regression 3, 8.mars
    Row(Trial=6,  AUC=0.9147), #Gradient Boosted Trees 2, 8.mars
    Row(Trial=7,  AUC=0.9114), #Logistic Regression 4, 12.mars
    Row(Trial=8,  AUC=0.9005), #Logistic Regression 5, 12.mars
    Row(Trial=9,  AUC=0.9237), #Logistic Regression 6, 12.mars
    Row(Trial=10, AUC=0.9350), #Logistic Regression 7, 12.mars
    Row(Trial=11, AUC=0.9411), #Random Forest 2, 12.mars
    Row(Trial=12, AUC=0.9156), #Gradient Boosted Trees 3, 12.mars
    Row(Trial=13, AUC=0.9240), #Random Forest 3, 15.mars
    Row(Trial=14, AUC=0.9360), #Gradient Boosted Trees 5, 17.mars
    Row(Trial=15, AUC=0.9379), #Multilayer Perceptron 1, 23.mars
    Row(Trial=16, AUC=0.9238), #Multilayer Perceptron 2, 29.mars
    Row(Trial=17, AUC=0.9136), #Multilayer Perceptron 3, 29.mars
    Row(Trial=18, AUC=0.9137), #Multilayer Perceptron 4, 29.mars
    Row(Trial=19, AUC=0.9034), #Multilayer Perceptron 5, 29.mars
    Row(Trial=20, AUC=0.8837), #Multilayer Perceptron 6, 29.mars
    Row(Trial=21, AUC=0.8847), #Multilayer Perceptron 7, 29.mars
    Row(Trial=22, AUC=0.9314), #Multilayer Perceptron 8, 29.mars
    Row(Trial=23, AUC=0.9452), #Multilayer Perceptron 9, 2.apríl
]

df_auc = spark.createDataFrame(data)
display(df_auc)

Databricks visualization. Run in Databricks to view.

In [0]:
import plotly.graph_objects as go

rows = df_auc.orderBy("Trial").collect()
trials = [r.Trial for r in rows]
aucs = [r.AUC for r in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=trials, y=aucs, mode="lines+markers", name="AUC", line=dict(color="#919191")))

# Lines with information about where LR and MLP stop/start
fig.add_vline(x=10, line_dash="dash", line_color="#BF7080",
              annotation_text="LR testing stops", annotation_position="top left")
fig.add_vline(x=16, line_dash="dash", line_color="#99DDB4",
              annotation_text="MLP testing starts", annotation_position="top left")

fig.update_layout(
    title="AUC vs Trials",
    xaxis_title="Trial",
    yaxis_title="AUC",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)
fig.show()


## MLP Fault Prediction Model — Results & Evaluation
This section presents the evaluation of the **Multilayer Perceptron (MLP)** classifier trained to predict X-ray controller faults using sensor telemetry data. The model uses a **failure horizon** label (fault within the next N hours) rather than the instantaneous fault state, enabling **early warning** capability.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score, accuracy_score

pdf = cached_pdf.copy()

y_true = pdf["horizon_failure"].values
y_pred = pdf["daily_prediction"].values

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# --- Figure: Confusion Matrix + Metrics side by side ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.1]})

# Left: Confusion Matrix
ax = axes[0]
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Confusion Matrix\n(Daily Predictions vs Horizon Label)", fontsize=13, fontweight="bold", pad=12)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["No Fault (0)", "Fault (1)"], fontsize=11)
ax.set_yticklabels(["No Fault (0)", "Fault (1)"], fontsize=11)
ax.set_xlabel("Predicted", fontsize=12, labelpad=8)
ax.set_ylabel("Actual", fontsize=12, labelpad=8)

# Annotate cells
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        label_map = {(0,0): "TN", (0,1): "FP", (1,0): "FN", (1,1): "TP"}
        ax.text(j, i, f"{label_map[(i,j)]}\n{cm[i, j]}",
                ha="center", va="center", fontsize=14, fontweight="bold", color=color)

# Right: Metrics summary
ax2 = axes[1]
ax2.axis("off")

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

metrics_text = (
    f"Accuracy:       {acc:.3f}\n"
    f"Precision:      {prec:.3f}\n"
    f"Recall:         {rec:.3f}\n"
    f"F1 Score:       {f1:.3f}\n"
    f"\n"
    f"True Positives:   {tp}\n"
    f"False Positives:  {fp}\n"
    f"True Negatives:   {tn}\n"
    f"False Negatives:  {fn}\n"
    f"\n"
    f"Total device-days: {len(pdf)}\n"
    f"Devices:           {pdf['properties_deviceId'].nunique()}\n"
    f"Date range:        {pdf['date'].min()} \u2192 {pdf['date'].max()}"
)

ax2.text(0.05, 0.95, "Classification Metrics", fontsize=13, fontweight="bold",
         va="top", transform=ax2.transAxes)
ax2.text(0.05, 0.82, metrics_text, fontsize=12, va="top",
         transform=ax2.transAxes, fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f4f8", edgecolor="#cccccc"))

plt.tight_layout()
plt.savefig("/tmp/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=["No Fault", "Fault"]))

In [0]:
import plotly.graph_objects as go
import pandas as pd

pdf = cached_pdf.copy()

# Short device labels (first 8 chars)
pdf["device_short"] = pdf["properties_deviceId"].str[:8] + "\u2026"

# Pivot: rows=devices, columns=dates, values=prediction
pivot = pdf.pivot_table(index="device_short", columns="date", values="daily_prediction", aggfunc="max")
pivot = pivot.fillna(-1)

# Also get actual fault events
fault_pivot = pdf.pivot_table(index="device_short", columns="date", values="actual_fault_state", aggfunc="max").fillna(0)

# Build custom text for hover
hover_text = []
for dev in pivot.index:
    row_text = []
    for d in pivot.columns:
        pred = int(pivot.loc[dev, d]) if pivot.loc[dev, d] >= 0 else "N/A"
        fault = int(fault_pivot.loc[dev, d]) if dev in fault_pivot.index and d in fault_pivot.columns else 0
        full_id = pdf[pdf["device_short"] == dev]["properties_deviceId"].iloc[0]
        row_text.append(f"Device: {full_id}<br>Date: {d}<br>Prediction: {pred}<br>Actual Fault: {fault}")
    hover_text.append(row_text)

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=[str(d) for d in pivot.columns],
    y=pivot.index,
    text=hover_text,
    hoverinfo="text",
    colorscale=[[0, "#d4edda"], [1, "#f5c6cb"]],
    showscale=False,
    zmin=0, zmax=1
))

# Mark actual fault events with a star
for i, dev in enumerate(pivot.index):
    for j, d in enumerate(pivot.columns):
        if dev in fault_pivot.index and d in fault_pivot.columns:
            if fault_pivot.loc[dev, d] == 1:
                fig.add_trace(go.Scatter(
                    x=[str(d)], y=[dev],
                    mode="markers",
                    marker=dict(symbol="star", size=18, color="black", line=dict(width=1.5, color="white")),
                    showlegend=False,
                    hoverinfo="skip"
                ))

fig.add_annotation(
    text="\u2605 = Actual fault event  |  <span style='color:#2d6a4f'>\u25a0</span> No fault predicted  |  <span style='color:#c9184a'>\u25a0</span> Fault predicted",
    xref="paper", yref="paper", x=0.5, y=-0.15,
    showarrow=False, font=dict(size=12), align="center"
)

fig.update_layout(
    title=dict(text="Daily Fault Predictions Across Fleet", font=dict(size=16)),
    xaxis_title="Date",
    yaxis_title="Device",
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=520,
    margin=dict(b=80),
    xaxis=dict(tickangle=-45, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10), autorange="reversed")
)
fig.show()

In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

pdf = cached_pdf.copy()

# Find devices with actual faults in the test period
fault_devices = pdf[pdf["actual_fault_state"] == 1]["properties_deviceId"].unique()

if len(fault_devices) > 0:
    target_device = fault_devices[0]
else:
    target_device = pdf[pdf["horizon_failure"] == 1]["properties_deviceId"].iloc[0]

device_pdf = pdf[pdf["properties_deviceId"] == target_device].sort_values("date").copy()
device_short = target_device[:12] + "\u2026"

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Model Prediction (daily max)",
        "Horizon Failure Label (ground truth)",
        "Actual Fault State"
    ),
    row_heights=[0.33, 0.33, 0.33]
)

# Prediction
fig.add_trace(go.Bar(
    x=device_pdf["date"], y=device_pdf["daily_prediction"],
    marker_color=["#e74c3c" if v == 1 else "#2ecc71" for v in device_pdf["daily_prediction"]],
    name="Prediction", showlegend=True
), row=1, col=1)

# Horizon failure label
fig.add_trace(go.Bar(
    x=device_pdf["date"], y=device_pdf["horizon_failure"],
    marker_color=["#e67e22" if v == 1 else "#95a5a6" for v in device_pdf["horizon_failure"]],
    name="Horizon Label", showlegend=True
), row=2, col=1)

# Actual fault state
fig.add_trace(go.Bar(
    x=device_pdf["date"], y=device_pdf["actual_fault_state"],
    marker_color=["#c0392b" if v == 1 else "#bdc3c7" for v in device_pdf["actual_fault_state"]],
    name="Actual Fault", showlegend=True
), row=3, col=1)

# Mark the actual fault day with vertical line
fault_days = device_pdf[device_pdf["actual_fault_state"] == 1]["date"]
for fd in fault_days:
    for row in [1, 2, 3]:
        fig.add_vline(x=str(fd), line_dash="dash", line_color="red", line_width=2, row=row, col=1)

fig.update_layout(
    title=dict(
        text=f"Fault Prediction Timeline \u2014 Device {device_short}",
        font=dict(size=16)
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=550,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5)
)

for i in range(1, 4):
    fig.update_yaxes(range=[-0.1, 1.3], tickvals=[0, 1], ticktext=["No", "Yes"], row=i, col=1)
    fig.update_xaxes(showgrid=True, gridcolor="#eeeeee", row=i, col=1)
    fig.update_yaxes(showgrid=False, row=i, col=1)

fig.show()

# Print summary
early_warn_days = device_pdf[
    (device_pdf["daily_prediction"] == 1) & 
    (device_pdf["actual_fault_state"] == 0) &
    (device_pdf["horizon_failure"] == 1)
]
print(f"\nDevice: {target_device}")
print(f"Fault occurred on: {list(fault_days)}")
print(f"Days model predicted fault BEFORE actual event: {len(early_warn_days)}")
print(f"\u2192 Early warning lead time: {len(early_warn_days)} day(s)")

In [0]:
import plotly.graph_objects as go

# Trial data with model types
trials = [
    (1,  0.6525, "LR"),  (2,  0.6251, "RF"),  (3,  0.5383, "GBT"),
    (4,  0.9115, "LR"),  (5,  0.9139, "LR"),  (6,  0.9147, "GBT"),
    (7,  0.9114, "LR"),  (8,  0.9005, "LR"),  (9,  0.9237, "LR"),
    (10, 0.9350, "LR"),  (11, 0.9411, "RF"),  (12, 0.9156, "GBT"),
    (13, 0.9240, "RF"),  (14, 0.9360, "GBT"), (15, 0.9379, "MLP"),
    (16, 0.9238, "MLP"), (17, 0.9136, "MLP"), (18, 0.9137, "MLP"),
    (19, 0.9034, "MLP"), (20, 0.8837, "MLP"), (21, 0.8847, "MLP"),
    (22, 0.9314, "MLP"), (23, 0.9452, "MLP"),
]

trial_nums = [t[0] for t in trials]
aucs = [t[1] for t in trials]
model_types = [t[2] for t in trials]

colors = {"LR": "#3498db", "RF": "#2ecc71", "GBT": "#e67e22", "MLP": "#9b59b6"}
full_names = {"LR": "Logistic Regression", "RF": "Random Forest", "GBT": "Gradient Boosted Trees", "MLP": "Multilayer Perceptron"}

fig = go.Figure()

# Connecting line
fig.add_trace(go.Scatter(
    x=trial_nums, y=aucs, mode="lines",
    line=dict(color="#cccccc", width=1.5),
    showlegend=False, hoverinfo="skip"
))

# Points colored by model type
for mt in ["LR", "RF", "GBT", "MLP"]:
    idx = [i for i, m in enumerate(model_types) if m == mt]
    fig.add_trace(go.Scatter(
        x=[trial_nums[i] for i in idx],
        y=[aucs[i] for i in idx],
        mode="markers",
        name=full_names[mt],
        marker=dict(color=colors[mt], size=10, line=dict(width=1, color="white")),
        hovertemplate="Trial %{x}<br>AUC: %{y:.4f}<extra>" + full_names[mt] + "</extra>"
    ))

# Highlight best trial
best_idx = aucs.index(max(aucs))
fig.add_annotation(
    x=trial_nums[best_idx], y=aucs[best_idx],
    text=f"Best: {max(aucs):.4f}",
    showarrow=True, arrowhead=2, arrowcolor="#333",
    ax=30, ay=-30,
    font=dict(size=12, color="#333", family="Arial Black"),
    bgcolor="#ffffcc", bordercolor="#333", borderwidth=1
)

# Phase annotations
fig.add_vrect(x0=0.5, x1=3.5, fillcolor="#f0f0f0", opacity=0.3, line_width=0,
              annotation_text="Initial\nbaseline", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=3.5, x1=14.5, fillcolor="#e8f4fd", opacity=0.2, line_width=0,
              annotation_text="Feature engineering\n& tuning", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")
fig.add_vrect(x0=14.5, x1=23.5, fillcolor="#f3e8fd", opacity=0.2, line_width=0,
              annotation_text="MLP\nexploration", annotation_position="top left",
              annotation_font_size=10, annotation_font_color="#888")

fig.update_layout(
    title=dict(text="Model Development: AUC Across Trials", font=dict(size=16)),
    xaxis=dict(
        title=dict(text="Trial", standoff=50),
        showgrid=True, gridcolor="#eeeeee", dtick=1, range=[0, 24]
    ),
    yaxis=dict(
        title="AUC (Area Under ROC Curve)",
        showgrid=True, gridcolor="#eeeeee", range=[0.5, 1.0]
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
    height=450
)
fig.show()

print(f"Best AUC: {max(aucs):.4f} (Trial {trial_nums[best_idx]}, {full_names[model_types[best_idx]]})")
print(f"AUC improvement from baseline: {max(aucs) - aucs[0]:.4f} (+{((max(aucs)/aucs[0])-1)*100:.1f}%)")


In [0]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

pdf = cached_pdf.copy()

# Compute per-device metrics
device_metrics = []
for dev_id, grp in pdf.groupby("properties_deviceId"):
    y_true = grp["horizon_failure"].values
    y_pred = grp["daily_prediction"].values
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    acc = (tp + tn) / len(grp) if len(grp) > 0 else 0
    has_fault = grp["actual_fault_state"].max()
    has_horizon = grp["horizon_failure"].max()
    device_metrics.append({
        "device": dev_id[:8] + "\u2026",
        "device_full": dev_id,
        "accuracy": acc,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "days": len(grp),
        "has_fault": has_fault,
        "has_horizon": has_horizon,
        "category": "Faulted" if has_fault else ("At-risk (horizon)" if has_horizon else "Healthy")
    })

df_metrics = pd.DataFrame(device_metrics).sort_values("accuracy")

cat_colors = {"Faulted": "#e74c3c", "At-risk (horizon)": "#e67e22", "Healthy": "#2ecc71"}

fig = go.Figure()

for cat in ["Faulted", "At-risk (horizon)", "Healthy"]:
    subset = df_metrics[df_metrics["category"] == cat]
    if len(subset) == 0:
        continue
    fig.add_trace(go.Bar(
        y=subset["device"],
        x=subset["accuracy"],
        orientation="h",
        name=cat,
        marker_color=cat_colors[cat],
        hovertemplate=(
            "Device: %{customdata[0]}<br>"
            "Accuracy: %{x:.1%}<br>"
            "TP: %{customdata[1]}, FP: %{customdata[2]}<br>"
            "TN: %{customdata[3]}, FN: %{customdata[4]}<extra></extra>"
        ),
        customdata=subset[["device_full", "TP", "FP", "TN", "FN"]].values
    ))

fig.update_layout(
    title=dict(text="Per-Device Prediction Accuracy", font=dict(size=16)),
    xaxis_title="Accuracy",
    yaxis_title="Device",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(range=[0, 1.05], tickformat=".0%", showgrid=True, gridcolor="#eeeeee"),
    yaxis=dict(tickfont=dict(size=10)),
    barmode="group",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)

# Add average line
avg_acc = df_metrics["accuracy"].mean()
fig.add_vline(x=avg_acc, line_dash="dash", line_color="#555",
              annotation_text=f"Fleet avg: {avg_acc:.1%}", annotation_position="top right")

fig.show()

print(f"Fleet average accuracy: {avg_acc:.1%}")
print(f"Devices with faults correctly caught: {df_metrics[df_metrics['has_fault'] == 1]['TP'].sum()} / {df_metrics[df_metrics['has_fault'] == 1][['TP','FN']].sum().sum()}")